# TradeLab ML Module Example

This notebook demonstrates how to use the newly added ML module in `trade_lab.ml`.

Workflow:
1. Download OHLCV data.
2. Build signal/indicator-based features.
3. Create ML targets.
4. Train a neural model with `MLTrainer`.
5. Run walk-forward validation.
6. Use the trained model in `MLStrategy` + `BacktestEngine`.

In [ ]:
from pathlib import Path
import sys

import pandas as pd
import yfinance as yf

# Allow running from repository root or from examples/.
ROOT = Path.cwd().resolve().parent if Path.cwd().name == "examples" else Path.cwd().resolve()
SRC = ROOT / "src"
if str(SRC) not in sys.path:
    sys.path.insert(0, str(SRC))

from trade_lab.signals import OHLC
from trade_lab.indicators import EMA, RSI
from trade_lab.ml import (
    FutureReturn,
    FeatureScaler,
    MLTrainer,
    dense_model,
)
from trade_lab.strategies import MLStrategy
from trade_lab.backtesting.engine import BacktestEngine

> If this import fails because TensorFlow/Keras is not installed, install optional ML deps first (e.g. `pip install -e .[ml]`).

In [ ]:
if MLTrainer is None:
    raise RuntimeError("MLTrainer is unavailable. Install ML dependencies: pip install -e .[ml]")

In [ ]:
# 1) Download data
ticker = "SPY"
start = "2020-01-01"
end = "2026-01-01"

df = yf.download(ticker, start=start, end=end)
if isinstance(df.columns, pd.MultiIndex):
    df.columns = df.columns.droplevel("Ticker")

df = df.dropna().copy()
print(f"Downloaded {len(df)} rows for {ticker}.")
df.tail()

## Build Features + Target

In [ ]:
# 2) Define feature pipeline (signals + indicators)
# OHLC signal is passed into EMA as an example of signal->indicator chaining.
ohlc_signal = OHLC()
indicators = [
    EMA(ohlc_signal, period=20),
    EMA(period=50),
    RSI(period=14),
]

# 3) Target: future return over 5 bars, squashed to [-1, 1]
target = FutureReturn(periods=5, column="Close", scale=10.0)

# 4) Model factory + scaler
model_builder = dense_model(layers=[64, 32], dropout=0.2, learning_rate=0.001)
scaler = FeatureScaler(method="standard")

trainer = MLTrainer(
    indicators=indicators,
    target=target,
    model_builder=model_builder,
    scaler=scaler,
)

In [ ]:
X, y, feature_columns, clean_index = trainer.build_dataset(df.copy())
print("Feature matrix shape:", X.shape)
print("Target shape:", y.shape)
print("First 10 feature columns:")
feature_columns[:10]

## Train Model

In [ ]:
trained = trainer.train(
    df=df.copy(),
    epochs=100,
    batch_size=32,
    validation_split=0.2,
    verbose=1,
)

print("Trained model ready for MLStrategy.")
print("Number of model input features:", len(trained.input_names))
trained.input_names[:10]

## Walk-Forward Validation

In [ ]:
wf_results = trainer.walk_forward(
    df=df.copy(),
    n_splits=3,
    expanding=True,
    initial_train_ratio=0.6,
    epochs=10,
    batch_size=32,
    verbose=0,
)

summary = pd.DataFrame({
    "fold": [r.fold for r in wf_results],
    "train_loss": [r.train_loss for r in wf_results],
    "n_test": [len(r.test_predictions) for r in wf_results],
})
summary

## Use Trained Model in Backtesting

In [ ]:
ml_strategy = MLStrategy(
    model=trained,
    indicators=indicators,
    allow_long=True,
    allow_short=True,
    entry_threshold=0.2,
    exit_threshold=0.05,
)

engine = BacktestEngine(
    strategy=ml_strategy,
    ticker=ticker,
    start="2024-01-01",
    end="2026-01-01",
    initial_capital=100_000,
    commission=0.001,
    slippage=0.0005,
)

result = engine.run()
result.metrics

In [ ]:
result.trade_log.tail()